# 📊 Adım 1: Keşifsel Veri Analizi (EDA) - Tam Analiz
Bu notebook, projenin temelini oluşturan 1 Milyon satırlık verinin istatistiksel ve görsel analizlerini içerir.

### 1. Kütüphaneler ve Spark Oturumu

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

spark = SparkSession.builder.appName("BankDatasetEDA").getOrCreate()

### 2. Veri Yükleme ve Ön İşleme

In [ ]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("/app/dataset/bankdataset.csv")
# Sutun isimlerini normalize et
df = df.withColumnRenamed("Date", "date").withColumnRenamed("Domain", "domain").withColumnRenamed("Location", "location").withColumnRenamed("Value", "value").withColumnRenamed("Transaction_count", "transaction_count")
print(f"Toplam Satir: {df.count()}")
df.printSchema()

### 3. İstatistiksel Özet

In [ ]:
df.select("value", "transaction_count").describe().show()

### 4. Domain ve Lokasyon Analizi

In [ ]:
domain_stats = df.groupBy("domain").agg(count("*").alias("count"), avg("value").alias("avg_value")).orderBy(desc("count"))
domain_stats.show()

location_stats = df.groupBy("location").agg(sum("value").alias("total_value")).orderBy(desc("total_value")).limit(10)
location_stats.show()

### 5. Zaman Serisi Analizi

In [ ]:
df = df.withColumn("parsed_date", to_date(col("date"), "yyyy-MM-dd"))
daily_trend = df.groupBy("parsed_date").agg(sum("value").alias("daily_total")).orderBy("parsed_date").toPandas()
plt.figure(figsize=(12,6))
plt.plot(daily_trend["parsed_date"], daily_trend["daily_total"], color="red", marker="o")
plt.title("Günlük Toplam İşlem Hacmi Trendi")
plt.xticks(rotation=45)
plt.show()